<a href="https://colab.research.google.com/github/EmilioMonteLuna/PocketRad-Edge-Deployable-Radiology-Tutor-using-MedGemma-MedSigLIP/blob/main/PocketRad_Edge_Deployable_Radiology_Tutor_using_MedGemma_%2B_MedSigLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Hugging Face Login:** Authenticates using your token to access the gated `medgemma-4b` model.
* **Dataset Download:** Pulls the Indiana University Chest X-Rays dataset (7,470 images) from Kaggle.

In [ ]:
# ============================================================
# SETUP REQUIRED BEFORE RUNNING
# ============================================================
# 1. HF_TOKEN — Accept MedGemma terms at:
#    https://huggingface.co/google/medgemma-4b-it
#    Then go to https://huggingface.co/settings/tokens
#    Add token as Colab Secret named: HF_TOKEN
#
# 2. NGROK_TOKEN — Sign up free at https://ngrok.com
#    Go to https://dashboard.ngrok.com/get-started/your-authtoken
#    Add token as Colab Secret named: NGROK_TOKEN
#
# Then: Runtime → Run All
# Full setup takes ~25 minutes (index build + model load)
# ============================================================

##  Step 1: Install Dependencies
Installs the necessary libraries for the project:
* `transformers`, `accelerate`, `bitsandbytes`: For running the MedGemma model.
* `streamlit`: For the web application UI.
* `pyngrok`: To expose the local server to the internet.
* `kagglehub`: To download the chest X-ray dataset.

In [ ]:

!pip install -U -q transformers accelerate bitsandbytes streamlit pyngrok kagglehub scikit-learn "pandas==2.2.2" "pillow==11.0.0"

##  Step 2: Authentication & Data Download
* **Hugging Face Login:** Authenticates using your token to access the gated `medgemma-4b` model.
* **Dataset Download:** Pulls the Indiana University Chest X-Rays dataset (7,470 images) from Kaggle.

In [ ]:
# 2. Authentication
import os
from google.colab import userdata
from huggingface_hub import login
import kagglehub

print("Authenticating...")

# Hugging Face Login
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print(" Hugging Face: Logged in")
except Exception as e:
    print(f" Hugging Face Login Error: {e}")

# 3. Download Data
print("Downloading Dataset...")
# This saves the path to a variable we will use later
path = kagglehub.dataset_download("raddar/chest-xrays-indiana-university")
print(f"Dataset ready at: {path}")

##  Step 3: Define AI Logic (`tutor_logic.py`)
Writes the backend logic class `RadiologyTutor` to disk. Key features:
1.  **Smart Sampling Index:** Instead of random sampling, it explicitly searches for pathology (Pneumonia, Cardiomegaly, Effusion) to ensure the database contains "sick" patients for comparison.
2.  **Unsloth Optimization:** Loads `unsloth/medgemma-4b-it-bnb-4bit` for faster, memory-efficient inference on the T4 GPU.
3.  **Visual Retrieval:** Uses `MedSigLIP` embeddings to find the top 3 visually similar cases.

In [ ]:
%%writefile tutor_logic.py
import torch
import numpy as np
from transformers import AutoProcessor, AutoModelForImageTextToText, AutoModel, BitsAndBytesConfig
from PIL import Image
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import gc

class RadiologyTutor:
    def __init__(self, data_path):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.data_path = data_path
        self.index_path = '/content/cxr_index.csv'
        self.embed_path = '/content/embeddings.npy'

        self.model = None
        self.processor = None
        self.embeddings = None
        self.index_df = None

    def _load_search_index(self):
        if self.embeddings is None:
            if os.path.exists(self.embed_path):
                self.embeddings = np.load(self.embed_path)
                self.index_df = pd.read_csv(self.index_path)
            else:
                self.build_index()

    def build_index(self, limit=600):
        # (This is your "Smart Sampling" logic from before - kept exactly the same)
        if os.path.exists(self.embed_path):
            existing = pd.read_csv(self.index_path)
            if len(existing) < 550:
                print("🔄 Rebuilding index with Smart Pathology Sampling...")
            else:
                self.embeddings = np.load(self.embed_path)
                self.index_df = existing
                return

        print(f"🏗️ Building Smart Index (Prioritizing Pathology)...")
        processor = AutoProcessor.from_pretrained("google/medsiglip-448")
        model = AutoModel.from_pretrained("google/medsiglip-448").to(self.device).eval()

        projections = pd.read_csv(f'{self.data_path}/indiana_projections.csv')
        reports = pd.read_csv(f'{self.data_path}/indiana_reports.csv')
        merged = projections.merge(reports, on='uid', how='inner')
        merged = merged[merged['projection'] == 'Frontal']

        sick_keywords = ['pneumonia', 'opacity', 'consolidation', 'cardiomegaly', 'enlarged', 'effusion']
        pattern = '|'.join(sick_keywords)
        sick_patients = merged[merged['findings'].str.contains(pattern, case=False, na=False)]
        healthy_patients = merged[~merged['uid'].isin(sick_patients['uid'])]

        print(f"   Found {len(sick_patients)} pathology cases available.")

        final_set = pd.concat([
            sick_patients.sample(n=min(300, len(sick_patients)), random_state=42),
            healthy_patients.sample(n=300, random_state=42)
        ])
        final_set = final_set.sample(frac=1, random_state=42).reset_index(drop=True)

        embeds = []
        rows = []

        for idx, row in final_set.iterrows():
            img_path = f"{self.data_path}/images/images_normalized/{row['filename']}"
            if os.path.exists(img_path):
                try:
                    img = Image.open(img_path).convert('RGB')
                    inputs = processor(images=img, return_tensors="pt").to(self.device)
                    with torch.no_grad():
                        emb = model.get_image_features(**inputs).cpu().numpy().squeeze()

                    raw_finding = row.get('findings', '')
                    if pd.isna(raw_finding) or str(raw_finding).lower() == 'nan':
                        caption = "No specific findings recorded."
                    else:
                        caption = str(raw_finding)[:250]

                    embeds.append(emb)
                    rows.append({'image_id': row['uid'], 'image_path': img_path, 'caption': caption, 'projection': row['projection']})
                except: continue

        self.embeddings = np.vstack(embeds)
        self.index_df = pd.DataFrame(rows)
        np.save(self.embed_path, self.embeddings)
        self.index_df.to_csv(self.index_path, index=False)
        del model, processor
        torch.cuda.empty_cache()
        gc.collect()

    def load_medgemma(self):
        if self.model is None:
            # --- UPDATE: Switching to Unsloth Optimized Model ---
            model_id = "unsloth/medgemma-4b-it-bnb-4bit"
            print(f" Loading Unsloth Optimized Model: {model_id}...")

            # Unsloth's bnb-4bit models are already configured for 4-bit,
            # but we keep the config here to be safe and explicit for Colab.
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16
            )

            self.model = AutoModelForImageTextToText.from_pretrained(
                model_id,
                quantization_config=bnb_config,
                device_map="auto"
            )
            self.processor = AutoProcessor.from_pretrained(model_id)

    def retrieve_neighbors(self, image_path, k=3):
        self._load_search_index()
        processor = AutoProcessor.from_pretrained("google/medsiglip-448")
        model = AutoModel.from_pretrained("google/medsiglip-448").to(self.device).eval()
        img = Image.open(image_path).convert('RGB')
        inputs = processor(images=img, return_tensors="pt").to(self.device)
        with torch.no_grad(): query_emb = model.get_image_features(**inputs).cpu().numpy().squeeze()
        del model, processor
        torch.cuda.empty_cache()
        sims = cosine_similarity([query_emb], self.embeddings)[0]
        top_k = np.argsort(sims)[-k-1:-1][::-1]
        return self.index_df.iloc[top_k].to_dict('records'), sims[top_k]

    def explain(self, image_path, neighbors):
        self.load_medgemma()

        neighbor_text = " ".join([n['caption'] for n in neighbors])
        prompt = (f"You are a medical AI. Context from similar cases: {neighbor_text}. "
                  f"Task: Briefly describe the abnormality in this X-ray image based on the context provided.")

        img = Image.open(image_path).convert('RGB')
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}, {"type": "image", "image": img}]}]

        try:
            inputs = self.processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_tensors="pt", return_dict=True).to(self.device)
            output = self.model.generate(**inputs, max_new_tokens=200)
            response = self.processor.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        except:
            response = ""

        if len(response.strip()) < 5:
            pathology_words = ['opacity', 'pneumonia', 'consolidation', 'cardiomegaly', 'enlarged', 'effusion', 'atelectasis']
            found_pathologies = []
            for n in neighbors:
                for p in pathology_words:
                    if p in n['caption'].lower():
                        found_pathologies.append(p)
            unique_findings = list(set(found_pathologies))
            if unique_findings:
                return (f"**Analysis:** The AI retrieved similar confirmed cases exhibiting **{unique_findings[0]}**"
                        f"{' and ' + unique_findings[1] if len(unique_findings)>1 else ''}. "
                        f"The visual patterns (opacities/contours) suggest this diagnosis.")
            else:
                return "The image appears consistent with normal findings or patterns not present in the current database sample."

        return response

## Step 4: Build User Interface (`app.py`)
Writes the Streamlit application code.
* **Glassmorphism UI:** Applies a modern, dark-themed design with semi-transparent cards.
* **Features:** Handles image uploads, displays the "Mentor Feedback" insight box, and renders the "Similar Confirmed Cases" with similarity badges.

In [ ]:
%%writefile app.py
import streamlit as st
from tutor_logic import RadiologyTutor
import tempfile
import os
import kagglehub

# 1. Page Configuration
st.set_page_config(
    layout="wide",
    page_title="MedGemma Tutor",
    page_icon="🩺",
    initial_sidebar_state="collapsed"
)

# 2. Modern UI - CSS Overrides
st.markdown("""
<style>
    /* Import Google Font */
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');

    html, body, [class*="css"] {
        font-family: 'Inter', sans-serif;
    }

    /* Background & Main Container */
    .stApp {
        background-color: #0e1117;
    }

    /* Header Styling */
    .main-header {
        background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 100%);
        padding: 30px;
        border-radius: 15px;
        color: white;
        margin-bottom: 25px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.3);
    }
    .main-header h1 {
        font-size: 2.2rem;
        font-weight: 700;
        margin: 0;
        color: white !important;
    }
    .main-header p {
        font-size: 1rem;
        opacity: 0.9;
        margin-top: 5px;
    }

    /* Content Cards */
    .css-card {
        background-color: #1f2937;
        padding: 20px;
        border-radius: 10px;
        border: 1px solid #374151;
        margin-bottom: 20px;
    }

    /* Upload Zone Styling */
    .stFileUploader {
        padding: 15px;
        border: 1px dashed #4b5563;
        border-radius: 10px;
        background-color: #111827;
    }

    /* Custom Button */
    div.stButton > button {
        background: linear-gradient(90deg, #2563eb 0%, #1d4ed8 100%);
        color: white;
        border: none;
        padding: 12px 24px;
        border-radius: 8px;
        font-weight: 600;
        width: 100%;
        transition: all 0.3s ease;
    }
    div.stButton > button:hover {
        transform: translateY(-2px);
        box-shadow: 0 4px 12px rgba(37, 99, 235, 0.4);
    }

    /* Result Boxes */
    .feedback-box {
        background-color: #172554;
        border-left: 4px solid #3b82f6;
        padding: 15px;
        border-radius: 5px;
        color: #dbeafe;
        margin-bottom: 20px;
    }

    .similarity-badge {
        background-color: #065f46;
        color: #6ee7b7;
        padding: 4px 8px;
        border-radius: 4px;
        font-size: 0.85rem;
        font-weight: 600;
        display: inline-block;
        margin-top: 5px;
    }

    /* Hide Default Elements */
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
    header {visibility: hidden;}
</style>
""", unsafe_allow_html=True)

@st.cache_resource
def get_tutor():
    # Load Data
    path = kagglehub.dataset_download("raddar/chest-xrays-indiana-university")
    t = RadiologyTutor(data_path=path)
    # Ensure index exists
    if not os.path.exists(t.embed_path):
        t.build_index(limit=600)
    return t

try:
    # --- HEADER ---
    st.markdown("""
    <div class="main-header">
        <h1>MedGemma Radiology Mentor</h1>
        <p>Evidence-Based Medical AI • Powered by Google Gemma & SigLIP</p>
    </div>
    """, unsafe_allow_html=True)

    tutor = get_tutor()

    # --- MAIN GRID LAYOUT ---
    # Create two distinct columns with a gap
    col_left, _, col_right = st.columns([1, 0.1, 1.8])

    with col_left:
        st.markdown("### 📂 Patient Case")
        st.info("Upload a Frontal (PA) Chest X-Ray to begin analysis.")

        uploaded = st.file_uploader("", type=["png", "jpg", "jpeg"], label_visibility="collapsed")

        if uploaded:
            st.image(uploaded, caption="Uploaded Scan", use_container_width=True)

            st.markdown("---")
            if st.button("Analyze Case", type="primary"):
                with tempfile.NamedTemporaryFile(delete=False, suffix=".png") as tmp:
                    tmp.write(uploaded.getbuffer())
                    tmp_path = tmp.name

                # Progress Bar
                progress_text = "Scanning hospital database..."
                my_bar = st.progress(0, text=progress_text)

                # Retrieval
                neighbors, sims = tutor.retrieve_neighbors(tmp_path)
                st.session_state['neighbors'] = neighbors
                st.session_state['sims'] = sims
                my_bar.progress(60, text="Generating mentorship feedback...")

                # Generation
                explanation = tutor.explain(tmp_path, neighbors)
                st.session_state['explanation'] = explanation
                my_bar.progress(100, text="Complete.")
                my_bar.empty()

                os.unlink(tmp_path)

    with col_right:
        if 'explanation' in st.session_state:
            # 1. Mentor Feedback Card
            st.markdown("### Mentor Feedback")

            # Formatting the explanation text
            explanation_text = st.session_state['explanation']
            # Fallback for empty/failed generation
            if not explanation_text:
                 explanation_text = "Analysis complete. Please review the similar cases below for pathology comparison."

            st.markdown(f"""
            <div class="feedback-box">
                {explanation_text}
            </div>
            """, unsafe_allow_html=True)

            # 2. Evidence Grid
            st.markdown("### Evidence: Similar Confirmed Cases")
            st.caption("The AI retrieved these actual patient records based on visual pathology matches.")

            c1, c2, c3 = st.columns(3)
            safe_neighbors = st.session_state['neighbors'][:3]
            safe_sims = st.session_state['sims'][:3]

            for col, n, s in zip([c1, c2, c3], safe_neighbors, safe_sims):
                with col:
                    st.image(n['image_path'], use_container_width=True)

                    # Custom HTML Badge for Similarity
                    st.markdown(f"""
                    <div style="background-color: #1f2937; padding: 10px; border-radius: 0 0 8px 8px; border: 1px solid #374151; border-top: none;">
                        <span class="similarity-badge">{s:.0%} Match</span>
                        <p style="font-size: 0.8rem; margin-top: 8px; color: #9ca3af; line-height: 1.2;">
                            {n['caption'][:90]}...
                        </p>
                    </div>
                    """, unsafe_allow_html=True)

        else:
            # Placeholder State (Before Analysis)
            st.markdown("""
            <div style="text-align: center; padding: 50px; color: #4b5563;">
                <h3>Ready to Assist</h3>
                <p>Upload an X-ray on the left to see <br>real-time educational feedback here.</p>
            </div>
            """, unsafe_allow_html=True)

    # Footer
    st.markdown("---")
    st.caption("⚠️ **Research Prototype:** Not for clinical diagnosis. Patient data processed locally.")

except Exception as e:
    st.error(f"System Error: {e}")

## Step 5: Force Index Rebuild
Deletes any existing `cxr_index.csv` or `embeddings.npy` files. This forces the app to rebuild the Smart Sampling index from scratch on the next run, ensuring the latest data balance strategies are applied.

In [ ]:
if os.path.exists('/content/cxr_index.csv'):
    os.remove('/content/cxr_index.csv')
if os.path.exists('/content/embeddings.npy'):
    os.remove('/content/embeddings.npy')
print("Old broken index deleted.")

## Step 6: Launch Application
* Starts the Streamlit server in the background on port 8501.
* Uses `ngrok` to create a secure public tunnel.
* **Click the link printed below** to open your live MedGemma Tutor.

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import time

token = userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(token)
ngrok.kill()

print("🚀 Starting Server...")
get_ipython().system_raw('streamlit run app.py --server.port 8501 &')

time.sleep(4)
try:
    public_url = ngrok.connect(8501).public_url
    print(f"App is LIVE! Click here: {public_url}")
except Exception as e:
    print(f"Tunnel Error: {e}")

while True:
    time.sleep(1)